# Machine Learning — Dự đoán mức độ căng thẳng (Stress Level)

**Mục tiêu:** Phân loại mức căng thẳng thành 3 nhóm (Low / Medium / High) từ dữ liệu đã tiền xử lý.

**Pipeline:**
1. Chuyển đổi nhãn (1–10) → 3 lớp
2. Stratified K-Fold (K=5) + SMOTE chỉ trên tập train từng fold
3. Huấn luyện Random Forest (baseline) và XGBoost (SOTA)
4. Đánh giá, lưu metrics, confusion matrix, SHAP, mô hình `.pkl`

---
## 1. Import thư viện & cấu hình đường dẫn

In [ ]:
import os
import warnings

import joblib
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# --- Đường dẫn (notebook nằm trong thư mục notebooks/) ---
PROC_PATH = '../data/processed/preprocessed_data.csv'
RAW_PATH = '../data/raw/Teen_Mental_Health_Dataset.csv'
METRICS_PATH = '../results/metrics/ml_metrics.csv'
FIG_MODELS_DIR = '../figures/models/'
FIG_SHAP_DIR = '../figures/explainability/'
MODEL_DIR = '../models/ml/'

for d in [METRICS_PATH.rsplit('/', 1)[0], FIG_MODELS_DIR, FIG_SHAP_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5
# Ngưỡng lệch lớp: max(count)/min(count) > ngưỡng → áp dụng SMOTE trên train fold
IMBALANCE_RATIO_THRESHOLD = 1.5
CLASS_NAMES = ['Low', 'Medium', 'High']

---
## 2. Tải dữ liệu & chọn đặc trưng (X)

- **X:** lấy từ `preprocessed_data.csv` (đã encode + scale).
- **y:** `stress_level` gốc (1–10) từ file raw — vì cột `stress_level` trong file processed đã bị **StandardScaler** nên không dùng để gán nhóm.

In [ ]:
df_proc = pd.read_csv(PROC_PATH)
df_raw = pd.read_csv(RAW_PATH)

assert len(df_proc) == len(df_raw), 'Số dòng processed và raw phải khớp nhau.'

# Đặc trưng đầu vào theo yêu cầu (Platform Usage = 3 cột one-hot)
FEATURE_COLS = [
    'age',
    'gender',
    'daily_social_media_hours',
    'platform_Both',
    'platform_Instagram',
    'platform_TikTok',
    'sleep_hours',
    'screen_time_before_sleep',
    'academic_performance',
    'physical_activity',
    'social_interaction_level',
]

missing_feats = [c for c in FEATURE_COLS if c not in df_proc.columns]
if missing_feats:
    raise ValueError(f'Thiếu cột trong preprocessed_data: {missing_feats}')

X = df_proc[FEATURE_COLS].copy()
stress_original = df_raw['stress_level'].values

print(f'Kích thước X: {X.shape}')
print(f'Stress level gốc — min: {stress_original.min()}, max: {stress_original.max()}')

---
## 3. Chuyển đổi biến mục tiêu (Target Transformation)

| Nhóm | Khoảng điểm |
|------|-------------|
| **Low** | 1 – 3 |
| **Medium** | 4 – 7 |
| **High** | 8 – 10 |

In [ ]:
def stress_to_class(level: int) -> str:
    """Ánh xạ điểm stress (1–10) sang nhãn 3 lớp."""
    if 1 <= level <= 3:
        return 'Low'
    if 4 <= level <= 7:
        return 'Medium'
    if 8 <= level <= 10:
        return 'High'
    raise ValueError(f'Giá trị stress_level không hợp lệ: {level}')


y_labels = np.array([stress_to_class(int(v)) for v in stress_original])

label_encoder = LabelEncoder()
label_encoder.fit(CLASS_NAMES)  # thứ tự cố định: Low=0, Medium=1, High=2
y = label_encoder.transform(y_labels)

dist = pd.Series(y_labels).value_counts().reindex(CLASS_NAMES)
print('Phân phối lớp mục tiêu:')
print(dist)
print(f"\nTỷ lệ lệch (max/min): {dist.max() / dist.min():.2f}")

---
## 4. Hàm tiện ích: SMOTE, metrics, vẽ Confusion Matrix

In [ ]:
def is_imbalanced(y_train: np.ndarray, threshold: float = IMBALANCE_RATIO_THRESHOLD) -> bool:
    """Kiểm tra mất cân bằng lớp trên tập train của fold."""
    counts = np.bincount(y_train, minlength=len(CLASS_NAMES))
    if counts.min() == 0:
        return True
    return counts.max() / counts.min() > threshold


def maybe_smote(X_train: pd.DataFrame, y_train: np.ndarray):
    """Áp dụng SMOTE CHỈ khi train fold bị lệch; tránh leakage vì không đụng tập test."""
    if is_imbalanced(y_train):
        smote = SMOTE(random_state=RANDOM_STATE)
        X_res, y_res = smote.fit_resample(X_train, y_train)
        return pd.DataFrame(X_res, columns=X_train.columns), y_res, True
    return X_train, y_train, False


def compute_metrics(y_true, y_pred, y_proba) -> dict:
    """Tính Accuracy, Precision, Recall, F1 (macro), ROC-AUC (macro OVR)."""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'roc_auc_macro': roc_auc_score(
            y_true, y_proba, multi_class='ovr', average='macro', labels=list(range(len(CLASS_NAMES)))
        ),
    }


def plot_confusion_matrix(cm, title: str, save_path: str):
    """Vẽ và lưu confusion matrix."""
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
    )
    ax.set_xlabel('Dự đoán')
    ax.set_ylabel('Thực tế')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 Đã lưu: {save_path}')

---
## 5. Định nghĩa mô hình & Stratified K-Fold Cross-Validation

Mỗi fold: SMOTE (nếu cần) → fit → đánh giá trên **test fold**. Metrics báo cáo là **trung bình** trên 5 fold.

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'XGBoost': xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=len(CLASS_NAMES),
        eval_metric='mlogloss',
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

fold_records = {name: [] for name in models}
oof_cm = {name: np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int) for name in models}
smote_applied_folds = 0

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_fit, y_train_fit, used_smote = maybe_smote(X_train, y_train)
    smote_applied_folds += int(used_smote)

    print(f'Fold {fold_idx}/{N_FOLDS} — train: {len(y_train_fit)}, test: {len(y_test)}, SMOTE: {used_smote}')

    for model_name, estimator in models.items():
        model = clone(estimator)
        model.fit(X_train_fit, y_train_fit)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        metrics = compute_metrics(y_test, y_pred, y_proba)
        metrics['fold'] = fold_idx
        fold_records[model_name].append(metrics)

        oof_cm[model_name] += confusion_matrix(
            y_test, y_pred, labels=list(range(len(CLASS_NAMES)))
        )

print(f'\nTổng số fold có áp dụng SMOTE: {smote_applied_folds}/{N_FOLDS}')

---
## 6. Tổng hợp metrics & lưu `ml_metrics.csv`

In [ ]:
metric_cols = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc_macro']

summary_rows = []
for model_name, records in fold_records.items():
    df_fold = pd.DataFrame(records)
    row = {'model': model_name}
    for col in metric_cols:
        row[col] = df_fold[col].mean()
        row[f'{col}_std'] = df_fold[col].std()
    summary_rows.append(row)

df_metrics = pd.DataFrame(summary_rows)
df_metrics = df_metrics.sort_values('f1_macro', ascending=False).reset_index(drop=True)

display_cols = ['model'] + metric_cols + [f'{c}_std' for c in metric_cols]
print('Bảng so sánh mô hình (trung bình 5-fold CV):')
display(df_metrics[display_cols])

df_metrics[display_cols].to_csv(METRICS_PATH, index=False)
print(f'\nĐã lưu metrics: {METRICS_PATH}')

best_model_name = df_metrics.loc[0, 'model']
print(f'\nMô hình tốt nhất theo F1-macro: {best_model_name}')

---
## 7. Confusion Matrix (tổng hợp out-of-fold trên toàn bộ CV)

In [ ]:
for model_name, cm in oof_cm.items():
    slug = model_name.lower().replace(' ', '_')
    save_path = os.path.join(FIG_MODELS_DIR, f'confusion_matrix_{slug}.png')
    plot_confusion_matrix(
        cm,
        title=f'Confusion Matrix (OOF) — {model_name}',
        save_path=save_path,
    )

---
## 8. Huấn luyện mô hình cuối & SHAP (giải thích)

Huấn luyện lại mô hình tốt nhất trên **toàn bộ** dữ liệu (áp dụng SMOTE nếu toàn cục bị lệch) để lưu `.pkl` và vẽ SHAP.

In [ ]:
X_final, y_final, used_smote_final = maybe_smote(X, y)
print(f'Huấn luyện cuối — mẫu: {len(y_final)}, SMOTE: {used_smote_final}')

best_estimator = clone(models[best_model_name])
best_estimator.fit(X_final, y_final)

model_slug = best_model_name.lower().replace(' ', '_')
model_path = os.path.join(MODEL_DIR, f'best_stress_classifier_{model_slug}.pkl')

artifact = {
    'model': best_estimator,
    'label_encoder': label_encoder,
    'class_names': CLASS_NAMES,
    'feature_cols': FEATURE_COLS,
    'best_model_name': best_model_name,
    'cv_f1_macro': float(df_metrics.loc[df_metrics['model'] == best_model_name, 'f1_macro'].iloc[0]),
}
joblib.dump(artifact, model_path)
print(f' Đã lưu mô hình: {model_path}')

In [ ]:
# SHAP TreeExplainer — dùng mẫu gốc (chưa SMOTE) để plot dễ đọc hơn
explainer = shap.TreeExplainer(best_estimator)
shap_values = explainer.shap_values(X)

# XGBoost / RF multiclass: list 3 ma trận (mỗi lớp) hoặc ndarray 3D
if isinstance(shap_values, list):
    shap_matrix = np.stack(shap_values, axis=-1)
else:
    shap_matrix = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_matrix,
    X,
    feature_names=FEATURE_COLS,
    class_names=CLASS_NAMES,
    show=False,
)
plt.tight_layout()
shap_path = os.path.join(FIG_SHAP_DIR, 'shap_importance.png')
plt.savefig(shap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Đã lưu SHAP summary: {shap_path}')

---
## 9. Tóm tắt

| Đầu ra | Đường dẫn |
|--------|-----------|
| Metrics | `results/metrics/ml_metrics.csv` |
| Confusion Matrix | `figures/models/confusion_matrix_*.png` |
| SHAP | `figures/explainability/shap_importance.png` |
| Mô hình | `models/ml/best_stress_classifier_*.pkl` |